# Analyze sequential images [Under development]
**# DO NOT USE**


This notebook demonstrates a powerful workflow leveraging **Google Cloud's BigQuery** and **Vertex AI Gemini** to perform analysis of Full scene Imagery Insights. The primary objective is to extract actionable insights from sequences of images, which is particularly valuable for logistics planning, site assessment, and identifying critical features at various locations.

### **Core Logic & Value Proposition:**

1.  **Data Acquisition from BigQuery**: The process begins by efficiently querying BigQuery to retrieve specific image URLs and associated metadata for identified 'tracks' (sequences of images).
2.  **Intelligent Data Preparation**: Image URLs are then transformed into `gs://` URIs, the optimized format for secure and high-performance access by Vertex AI services.
3.  **Multimodal AI Analysis with Gemini**: The heart of the solution lies in utilizing the advanced capabilities of the Gemini multimodal model. It processes sequences of images (representing a continuous view) to detect and describe specific features relevant to logistics drivers, such as:
    *   Obstructions to driveway or street entry.
    *   Presence of restrictive signage.
    *   Details about gated entries.
4.  **Actionable Insights**: The AI's structured analysis provides concise, driver-centric summaries, highlighting critical information that can impact delivery routes, access, and overall operational efficiency.
5.  **Clear Visualization**: Finally, the notebook presents the AI-generated insights alongside the actual images, allowing for immediate visual verification and deeper understanding by the customer.

This automated workflow empowers customers to quickly gain crucial intelligence from vast amounts of imagery data, reducing manual inspection time and improving decision-making for real-world logistical challenges.


In [1]:
# @title 1. Configuration & Environment Setup

# Project and Data Configuration
project_id = "YOUR_PROJECT_ID" #@param {type:"string"} #@param {type:"string"}
dataset_id = 'imagery_insights___us' #@param {type:"string"} #@param {type:"string"}
observations_table_name = 'pano_observations_latest' #@param {type:"string"}
urls_table_name = 'urls_new' #@param {type:"string"}

# Vertex AI Model Configuration
location = "global" #@param {type:"string"}
# Note: Using 'gemini-3.5-flash' for broad availability across projects
model_name = "gemini-3.5-flash" #@param {type:"string"}

# Specific Track IDs to analyze for this demonstration
track_ids_to_query = [
    't1:6mzYhoifk03FSDhXsTsykw:5001ee',
    't1:pWQRnW-wKvf-eLvaD___DQ:5001ee',
    't1:k1XEbQBR9Qp5Jp9mQgbp4Q:5001ee',
    't1:vAI8V0nzwZ3NnhC8Am4VFg:5001ee',
    't1:PqvE3_XZ7qZ0k01WqCL4Pw:5001ee'
]

print(f"[INFO] Environment configured. Using model: {model_name}")

[INFO] Environment configured. Using model: gemini-3.5-flash


In [2]:
# @title 2. Data Extraction from BigQuery
# @markdown Fetch image metadata and signed URLs by joining track information with asset records.

from google.cloud import bigquery
import pandas as pd

client = bigquery.Client(project=project_id)

# Prepare track IDs for SQL IN clause
track_ids_str = ", ".join([f"'{tid}'" for tid in track_ids_to_query])

query = f"""
    SELECT
        urls.signedUrl,
        obs.trackId
    FROM
        `{project_id}.{dataset_id}.{observations_table_name}` AS obs
    JOIN
        `{project_id}.{dataset_id}.{urls_table_name}` AS urls
    ON
        obs.observation0 = urls.observationId
    WHERE
        obs.trackId IN ({track_ids_str})
"""

print("Executing BigQuery query...")
df_results = client.query(query).to_dataframe()

if not df_results.empty:
    print(f"Successfully retrieved {len(df_results)} images.")
    display(df_results.head())
else:
    print("No data found for the provided Track IDs.")

Executing BigQuery query...


Successfully retrieved 5 images.


,signedUrl,trackId
0,https://storage.mtls.cloud.google.com/everythi...,t1:6mzYhoifk03FSDhXsTsykw:5001ee
1,https://storage.mtls.cloud.google.com/everythi...,t1:pWQRnW-wKvf-eLvaD___DQ:5001ee
2,https://storage.mtls.cloud.google.com/everythi...,t1:k1XEbQBR9Qp5Jp9mQgbp4Q:5001ee
3,https://storage.mtls.cloud.google.com/everythi...,t1:vAI8V0nzwZ3NnhC8Am4VFg:5001ee
4,https://storage.mtls.cloud.google.com/everythi...,t1:PqvE3_XZ7qZ0k01WqCL4Pw:5001ee


In [3]:
# @title 3. Pre-processing: URI Transformation
# @markdown Vertex AI optimizes performance when using native `gs://` URIs instead of signed HTTP URLs.

def convert_to_gs_uri(url):
    """Converts GCS HTTP URLs to native gs:// format."""
    if not isinstance(url, str): return url

    mapping = {
        'https://storage.googleapis.com/': 'gs://',
        'https://storage.mtls.cloud.google.com/': 'gs://'
    }

    for prefix, gs_prefix in mapping.items():
        if prefix in url:
            return url.replace(prefix, gs_prefix)
    return url

if 'df_results' in locals() and not df_results.empty:
    df_results['gs_url'] = df_results['signedUrl'].apply(convert_to_gs_uri)
    print("URIs converted successfully for Vertex AI compatibility.")
    display(df_results[['trackId', 'gs_url']].head())

URIs converted successfully for Vertex AI compatibility.


,trackId,gs_url
0,t1:6mzYhoifk03FSDhXsTsykw:5001ee,gs://everything_imagery/full_scene_home_depot/...
1,t1:pWQRnW-wKvf-eLvaD___DQ:5001ee,gs://everything_imagery/full_scene_home_depot/...
2,t1:k1XEbQBR9Qp5Jp9mQgbp4Q:5001ee,gs://everything_imagery/full_scene_home_depot/...
3,t1:vAI8V0nzwZ3NnhC8Am4VFg:5001ee,gs://everything_imagery/full_scene_home_depot/...
4,t1:PqvE3_XZ7qZ0k01WqCL4Pw:5001ee,gs://everything_imagery/full_scene_home_depot/...


## Analyze Images with Gemini



## Prepare Image URLs for Vertex AI

### Subtask:
Convert the `modified_url`s from the current `https://storage.mtls.cloud.google.com/` format to `gs://` format, which is required for `Part.from_uri()` in Vertex AI.


In [4]:
# @title 4. Verify Data for Analysis
# @markdown This cell performs a quick check on the prepared data to confirm the number of images
# @markdown available for each `trackId` before sending them to the AI model.
# @markdown This helps in understanding the scope of the analysis.

# Check if df_results contains data before grouping and counting.
if 'df_results' in locals() and not df_results.empty:
    # Group the DataFrame by 'trackId' and count the number of rows (images) in each group.
    track_counts = df_results.groupby('trackId').size()

    print("Images available per Track ID:")
    print(track_counts)

    # Determine the total number of unique tracks that will be analyzed.
    unique_tracks = df_results['trackId'].unique()
    print(f"\nTotal unique tracks to analyze: {len(unique_tracks)}")
else:
    print("No data available to verify. Please check previous steps.")

Images available per Track ID:
trackId
t1:6mzYhoifk03FSDhXsTsykw:5001ee    1
t1:PqvE3_XZ7qZ0k01WqCL4Pw:5001ee    1
t1:k1XEbQBR9Qp5Jp9mQgbp4Q:5001ee    1
t1:pWQRnW-wKvf-eLvaD___DQ:5001ee    1
t1:vAI8V0nzwZ3NnhC8Am4VFg:5001ee    1
dtype: int64

Total unique tracks to analyze: 5


## Analyze Images with Vertex AI Gemini 3 Flash

### Subtask:
Initialize Vertex AI and load the `gemini-3-flash-preview` model. Iterate through the prepared `gs://` image URIs, group them by `trackId`, and pass each sequence of images to the Vertex AI Gemini 3 Flash model for analysis, using `Part.from_uri()` for image inputs.


In [5]:
# @title 4. Multimodal Analysis with Vertex AI Gemini
# @markdown This cell passes image sequences to the AI model to identify specific logistical features.

!pip install -q -U google-genai

from google import genai
from google.genai import types
import time

# Initialize the Google Gen AI client for Vertex AI
client_genai = genai.Client(vertexai=True, project=project_id, location=location)

# Define the instructions for the Gemini model
analysis_prompt = """
Analyze this sequence of street view images for a logistics driver context.
Identify and describe the following features:
- **Obstructions to driveway entry / fences**: Describe any physical barriers.
- **Obstructions to street entry / Roadblocks**: Note any road blocks.
- **Signage**: Look for vehicle size/weight restrictions.
- **Gated entry**: State if a gate is present and its status (open/closed).

Provide a structured summary. If a feature is not present, state 'None'.
"""

vertex_responses = []

if 'df_results' in locals() and not df_results.empty:
    # Group images by trackId to process them as a sequence
    grouped = df_results.groupby('trackId')

    for track_id, group in grouped:
        print(f"Processing Track: {track_id}...")

        # Create the multimodal payload (Images + Text Instruction)
        content_parts = [
            types.Part.from_uri(file_uri=row['gs_url'], mime_type="image/jpeg")
            for _, row in group.iterrows() if row['gs_url'].startswith("gs://")
        ]
        content_parts.append(types.Part.from_text(text=analysis_prompt))

        try:
            # Generate content using the Gemini model
            response = client_genai.models.generate_content(
                model=model_name,
                contents=content_parts,
                config=types.GenerateContentConfig(
                    temperature=0.1,  # Lower temperature for more factual analysis
                    media_resolution=types.MediaResolution.MEDIA_RESOLUTION_HIGH
                )
            )
            vertex_responses.append({'trackId': track_id, 'response': response.text})
        except Exception as e:
            print(f"[ERROR] Failed to process {track_id}: {e}")

    print("\n[SUCCESS] Sequence analysis complete.")
else:
    print("[WARN] No data available for analysis. Please run step 2 and 3.")

/bin/bash: pip: command not found


Processing Track: t1:6mzYhoifk03FSDhXsTsykw:5001ee...


Processing Track: t1:PqvE3_XZ7qZ0k01WqCL4Pw:5001ee...


Processing Track: t1:k1XEbQBR9Qp5Jp9mQgbp4Q:5001ee...


Processing Track: t1:pWQRnW-wKvf-eLvaD___DQ:5001ee...


Processing Track: t1:vAI8V0nzwZ3NnhC8Am4VFg:5001ee...



[SUCCESS] Sequence analysis complete.


In [6]:
# @title 5. Final Report & Visualization
# @markdown Display the AI-generated insights alongside the source imagery for verification.

from IPython.display import Markdown, display, Image

if vertex_responses:
    display(Markdown("## Fleet Logistics Insight Report"))

    for entry in vertex_responses:
        display(Markdown(f"### Track ID: `{entry['trackId']}`"))

        # Show thumbnails for context
        img_row = df_results[df_results['trackId'] == entry['trackId']].iloc[0]
        display(Image(url=img_row['signedUrl'], width=400))

        display(Markdown("**AI Analysis Results:**"))
        display(Markdown(entry['response']))
        display(Markdown("---"))
else:
    print("No results to display.")

## Fleet Logistics Insight Report

### Track ID: `t1:6mzYhoifk03FSDhXsTsykw:5001ee`

**AI Analysis Results:**

Based on the provided street view image, here is the analysis for a logistics driver context:

*   **Obstructions to driveway entry / fences**: There is a chain-link fence visible in the background running along the property line/yard boundary near the trees, but it does not obstruct the immediate roadside or any visible driveway. There are no direct obstructions to property access from the main road.
*   **Obstructions to street entry / Roadblocks**: None. The concrete road is clear and open.
*   **Signage**: None. There are no visible traffic signs, weight limits, or vehicle size restriction signs.
*   **Gated entry**: None.

---

...

---

### Track ID: `t1:vAI8V0nzwZ3NnhC8Am4VFg:5001ee`

**AI Analysis Results:**

Based on the street view image provided, here is the analysis for a logistics driver context:

*   **Obstructions to driveway entry / fences**: There is a continuous stone/brick wall running along the right side of the sidewalk, acting as a boundary fence for the residential property. Further down, there is a black metal fence enclosing the apartment/residential community area.
*   **Obstructions to street entry / Roadblocks**: None. The concrete roadway is clear and unobstructed.
*   **Signage**: There is a parking restriction sign on the grass strip next to the sidewalk. It reads: 
    *   "NO PARKING STOPPING STANDING 7:15AM - 8:30AM 2:45PM - 3:30PM SCHOOL DAYS" with a left-pointing arrow. 
    *   *Note: There are no vehicle size, height, or weight restriction signs visible.*
*   **Gated entry**: None visible along the immediate roadside.

---